In [ ]:
## Imports

import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[1])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

from sigpy import mri
import scipy
import pickle
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap
import seaborn as sns
import sigpy as sp
import cupy as cp
import numpy as np
from sigpy.mri.app import TotalVariationRecon, L1WaveletRecon
from scipy.io import savemat
import twixtools
import matplotlib.pyplot as plt
from scipy.signal import medfilt
from scipy.signal import butter,filtfilt


## My files
from resp_signal_functions import resp_signal_all_slices, resp_signal_single_slice, resp_signal_center_sample_single_slice
from resp_signal_plot_functions import *
import gating_functions
import gating_visuals
from save_data_helpers import *
import recon_plot_helpers

## check shapes

In [ ]:
num_spokes = 200
base_dir = f'/data/lilianae/Subject3_MID0283_data_for_ADMM/{num_spokes}sp/'

dcfs = read_pickle(os.path.join(base_dir, 'dcf_bins.pkl'))
spoke_bins = read_pickle(os.path.join(base_dir, 'spoke_bins.pkl'))
data_bins = read_pickle(os.path.join(base_dir, 'data_bins.pkl'))
mps = read_pickle(os.path.join(base_dir, 'mps_nufft_adj_thresh_0.02.pkl'))
print(f'dcfs.shape = {dcfs[0].shape}')
print(f'data_bins[0].shape = {data_bins[0].shape}')
print(f'spoke_bins[0].shape = {spoke_bins[0].shape}')
print(f'mps.shape = {mps.shape}')

In [ ]:
gated_images = read_pickle(os.path.join(base_dir, 'nufft_images_bins.pkl'))
print(f'gated_images.shape = {gated_images[0].shape}')

In [ ]:
from pathlib import Path

# 1. Setup the directory
output_recon_dir = Path(f'/home/lilianae/projects/naf_clean/recons/subject3_mid0283/figs/{num_spokes}sp')
output_recon_dir.mkdir(parents=True, exist_ok=True) 

gated_recons = []

for i in range(len(gated_images)):
    gate_image = recon_plot_helpers.calculate_rss_image(gated_images[i])
    
    # 2. Refined Center Crop (assuming 512x512 input)
    # Using 512 instead of 511 for exact centering
    target_size = 256
    start_h = (gate_image.shape[1] - target_size) // 2  # 128
    start_w = (gate_image.shape[2] - target_size) // 2  # 128

    gate_image_cropped = gate_image[:, start_h:start_h+target_size, start_w:start_w+target_size]

    # 3. Construct the full output path
    # Path objects handle joining with the '/' operator beautifully
    save_path = output_recon_dir / f'rss_gate{i}.png'

    # 4. Call the function with save_fig=True
    fig, axs = recon_plot_helpers.plot_recons_all_axes(
        gate_image_cropped, 
        title=f"RSS Gate {i} (NUFFT) - MID 0283",
        save_fig=True,           # This MUST be True to trigger the save logic
        output_dir=str(save_path) # Convert Path object to string for plt.savefig
    )
    
    gated_recons.append(gate_image_cropped)

In [ ]:
# gated_recons = []
# output_recon_dir = Path(f'/home/lilianae/projects/naf_clean/recons/subject3_mid0283/figs/{num_spokes}sp')
# if not output_recon_dir.exists():
#     output_recon_dir.mkdir(parents=True, exist_ok=True) 

# for i in range(len(gated_images)):
#     gate_image = recon_plot_helpers.calculate_rss_image(gated_images[i])
#     # Center crop from 512x512 to 256x256
#     # Calculate the starting indices for the crop
#     start_h = (511 - 256) // 2  # 128
#     start_w = (511 - 256) // 2  # 128

#     # Crop along the last two dimensions
#     gate_image_cropped = gate_image[:, start_h:start_h+256, start_w:start_w+256]


#     fig, axs = recon_plot_helpers.plot_recons_all_axes(gate_image_cropped, title=f"RSS Gate {i} (NUFFT)- MID 0283 - 512 Readouts pts cropped",
#                                                        save_fig=False,
#                                                        output_dir=Path(output_recon_dir/f'rss_gate{i}.png'))
#     gated_recons.append(gate_image_cropped)


In [ ]:
# names = ["sagittal_5gates", "axial_5gates", "coronal_5gates"]
# slices = [44, 128, 128]

# for i, (name, slice_idx) in enumerate(zip(names, slices)):  
#     recon_plot_helpers.make_gif(gated_recons, slice_axis=i, slice_idx=slice_idx, gif_name=f"{name}", duration=100.0)